# Thu thập dữ liệu thời tiết từ Visual Crossing

Notebook này thực hiện thu thập dữ liệu khí tượng chi tiết theo giờ cho các thành phố đại diện tại Việt Nam.

**Chi tiết thu thập:**
- **Nguồn:** Visual Crossing Weather API.
- **Chiến lược:** Sử dụng danh sách nhiều API keys để vượt qua giới hạn quota hằng ngày của gói miễn phí.
- **Phương pháp:** Thu thập ngược từ ngày kết thúc về ngày bắt đầu theo các batch 40 ngày.
- **Sản phẩm:** Các file CSV lưu tại `data/raw/Weather_Hanoi_Raw/` (tương tự cho các thành phố khác).

In [ ]:
import os
import time
from datetime import datetime, timedelta
from pathlib import Path

import requests
import pandas as pd


def find_repo_root(marker: str = 'data') -> Path:
    """Tìm thư mục gốc của repo dựa trên marker."""
    current = Path.cwd()
    for parent in [current] + list(current.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError('Không tìm thấy repo root.')


ROOT = find_repo_root()
os.chdir(ROOT)
print(f'[OK] Working directory: {ROOT}')


In [ ]:
def crawl_weather_chain(
    list_keys: list[str],
    end_date_str: str,
    location: str = "Hanoi, Vietnam"
) -> None:
    """
    Thu thập dữ liệu thời tiết từ Visual Crossing API theo chuỗi (chain).

    Args:
        list_keys: Danh sách các API keys để vượt quota.
        end_date_str: Ngày kết thúc thu thập (YYYY-MM-DD).
        location: Tên địa điểm (ví dụ: "Hanoi, Vietnam").
    """
    current_end_date = datetime.strptime(end_date_str, '%Y-%m-%d')

    # Tự động tạo tên folder dựa trên location
    # VD: "Hanoi, Vietnam" -> "Weather_Hanoi_Raw"
    city_name = location.split(',')[0].replace(' ', '_')
    output_dir = Path(f"data/raw/Weather_{city_name}_Raw")

    if not output_dir.exists():
        output_dir.mkdir(parents=True, exist_ok=True)
        print(f"Starting weather data collection for {location}")

    print(f"Available API keys: {len(list_keys)}")
    print("-" * 40)

    for i, api_key in enumerate(list_keys):
        start_date_obj = current_end_date - timedelta(days=40)
        start_str = start_date_obj.strftime('%Y-%m-%d')
        end_str = current_end_date.strftime('%Y-%m-%d')

        print(f"Using key {i+1}: {api_key[:5]}*** | Range: {start_str} to {end_str}")
        url = f"https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/{location}/{start_str}/{end_str}"
        params = {
            'unitGroup': 'metric',
            'key': api_key,
            'contentType': 'csv',
            'include': 'hours'
        }
        try:
            response = requests.get(url, params=params, timeout=20)
            if response.status_code == 200:
                filename = f"{city_name}_{start_str}_to_{end_str}.csv"
                file_path = output_dir / filename
                with open(file_path, 'wb') as f:
                    f.write(response.content)
                print(f"Success: Saved {filename}")
                current_end_date = start_date_obj - timedelta(days=1)
            elif response.status_code == 429:
                print(f"Key {i+1} quota exceeded. Switching to next key...")
            else:
                print(f"Error {response.status_code} at key {i+1}.")
        except Exception as e:
            print(f"System error: {e}")

        time.sleep(2)
        print("-" * 20)

    print("Weather data collection completed.")

In [ ]:
# Thay the list_keys bang danh sach API keys thuc te
API_KEYS = ['YOUR_KEY_1', 'YOUR_KEY_2']  # each key covers ~40 days

crawl_weather_chain(API_KEYS, end_date_str='2026-03-10', location='Hanoi, Vietnam')
crawl_weather_chain(API_KEYS, end_date_str='2026-03-10', location='Da Nang, Vietnam')
crawl_weather_chain(API_KEYS, end_date_str='2026-03-10', location='Ho Chi Minh City, Vietnam')
